# SCARF on Google Colab for SCP3357

This notebook clones `HopTF`, installs Colab-compatible SCARF dependencies, downloads the official SCARF model assets, and computes `obsm['X_scarf']` for an SCP3357 `.h5ad` file.

Notes:
- Use a **GPU** Colab runtime.
- The SCARF bootstrap step downloads the official `model_files.zip` bundle from Zenodo, so expect a multi-gigabyte transfer.
- You can either build the SCP3357 `.h5ad` from a Single Cell Portal download or start from an existing `.h5ad` that already has `layers['counts']`.


In [ ]:
REPO_URL = "https://github.com/cpudan/HopTF.git"
REPO_REF = "dan"  # Change this if you want a different branch.
WORKDIR = "/content/HopTF"

MOUNT_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/HopTF"

# Option A: build from an SCP3357 portal bundle already stored somewhere Colab can read.
SCP3357_INPUT_DIR = None  # Example: "/content/drive/MyDrive/SCP3357"

# Option B: skip the build step and start from an existing AnnData file.
INPUT_H5AD = None  # Example: "/content/drive/MyDrive/HopTF/SCP3357_raw_counts.h5ad"

RAW_H5AD = f"{WORKDIR}/data/processed/scp3357/SCP3357_raw_counts.h5ad"
OUTPUT_H5AD = f"{WORKDIR}/data/processed/scp3357/SCP3357_raw_counts_scarf.h5ad"
COPY_OUTPUT_TO_DRIVE = True


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


In [ ]:
import os
import shutil
import subprocess

if os.path.isdir(WORKDIR):
    shutil.rmtree(WORKDIR)

subprocess.run([
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    WORKDIR,
], check=True)
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())


In [ ]:
import subprocess

subprocess.run([
    "bash",
    "scripts/bootstrap_scarf_colab.sh",
], check=True)


## Prepare the SCP3357 AnnData input

Choose one of the following:
- Set `INPUT_H5AD` above if you already have a `.h5ad` with `layers['counts']`.
- Set `SCP3357_INPUT_DIR` above if you want to build the `.h5ad` from the Single Cell Portal bundle inside Colab.


In [ ]:
import os
import subprocess

if INPUT_H5AD is None:
    if SCP3357_INPUT_DIR is None:
        raise ValueError("Set either INPUT_H5AD or SCP3357_INPUT_DIR before running this cell.")
    subprocess.run([
        "python",
        "scripts/build_scp3357_h5ad.py",
        "--input-dir",
        SCP3357_INPUT_DIR,
        "--output",
        RAW_H5AD,
    ], check=True)
    input_h5ad = RAW_H5AD
else:
    if not os.path.exists(INPUT_H5AD):
        raise FileNotFoundError(INPUT_H5AD)
    input_h5ad = INPUT_H5AD

print("Input AnnData:", input_h5ad)


In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Switch Colab to a GPU runtime before computing SCARF embeddings.")

subprocess.run([
    "python",
    "scripts/add_scarf_embeddings.py",
    "--input-h5ad",
    input_h5ad,
    "--output-h5ad",
    OUTPUT_H5AD,
    "--device",
    "cuda",
], check=True)


In [ ]:
import os
import shutil

import anndata as ad

adata = ad.read_h5ad(OUTPUT_H5AD, backed="r")
print("shape:", adata.shape)
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("X_scarf shape:", adata.obsm["X_scarf"].shape)
adata.file.close()

if COPY_OUTPUT_TO_DRIVE and MOUNT_DRIVE:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    destination = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(OUTPUT_H5AD))
    shutil.copy2(OUTPUT_H5AD, destination)
    print("Copied output to:", destination)
